# ML-03 — ML Task Framing

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ADHIRAJ994/Fly-Rank/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This notebook maps Lane 2 (Refresh / Content Opportunity Scoring) onto the ML loop. Sections in order: task type → target/proxy → success metric → unit of analysis → why ML beats a fixed rule.

## 1. My lane as an ML task (type)

**Lane:** Lane 2 — Refresh / Content Opportunity Scoring

**Task type: Supervised binary classification with ranking output**

The core task is to assign each eligible content page a probability score — the model's estimate that this page is a high-priority review candidate. Those probabilities are then used to rank pages from most to least urgent.

This is **classification** because the underlying label is binary: a page either needs priority attention (positive) or it doesn't (negative). But the output the user actually sees is a **ranked queue**, ordered by that probability — so evaluation uses ranking metrics (Precision@K, average precision), not just accuracy.

This is **supervised** because we define a label from observable data (what happened to traffic over a time window), train on that signal, and validate on held-out clients the model never saw during training.

**Why not clustering?** Clustering finds groups but doesn't rank. A reviewer needs an ordered list, not a label.

**Why not pure scoring/rules?** A fixed score (like the baseline) can rank, but it treats all signals with static weights and can't learn which combinations of signals actually predict future decline. The starter results prove this gap is real: Precision@50 jumps from 0.240 (baseline) to 0.740 (random forest).

In [2]:
# Load the starter data and confirm the task setup
import pandas as pd
import numpy as np

df = pd.read_csv('../../data/raw/content_refresh_anonymized.csv')

print("=== Starter dataset overview ===")
print(f"Total rows:    {len(df):,}")
print(f"Total columns: {df.shape[1]}")
print()
print("Task type: Supervised binary classification → ranked output")
print("Input:     Observable search + engagement signals (features)")
print("Output:    Probability score per page → ranked review queue")

=== Starter dataset overview ===
Total rows:    30,000
Total columns: 44

Task type: Supervised binary classification → ranked output
Input:     Observable search + engagement signals (features)
Output:    Probability score per page → ranked review queue


## 2. Target or proxy

**Proxy label (starter):** `is_declining = (trend_direction == 'down')`

This is the label used in the starter pipeline. It is a **proxy** — a stand-in for the outcome we actually care about — not a future-looking outcome. It answers: *is this page currently in a downward trend?* That's useful for building and testing the pipeline end-to-end, but it has a known weakness: the trend is calculated from the same window as the features, which creates a risk of the label encoding information the features already contain.

**Stronger capstone label (target direction):**
```
features from prior 90 days → decline over the next 30 days
```
This separates the feature window from the target window cleanly. A page's signals from the past 90 days are used to predict what happens in the *next* 30 days — a genuinely forward-looking label. This is the direction for the full capstone once the warehouse data is loaded.

**What 'positive' means:** A page is a positive example if it shows a meaningful, sustained drop in impressions or clicks over the target window — not a noise blip, not seasonality, not a page that simply has low volume. Minimum-volume filters (e.g. `impressions_90d >= 100`) keep noise out.

**What 'negative' means:** A page that held steady or grew over the same window — not that it was never worth reviewing, just that it was not declining in this period.

In [3]:
import pandas as pd
import numpy as np

df = pd.read_csv('../../data/raw/content_refresh_anonymized.csv')

# Define the proxy label
df['is_declining'] = (df['trend_direction'] == 'down').astype(int)

print("=== Target / proxy label: is_declining ===")
print()
label_counts = df['trend_direction'].value_counts()
print("trend_direction distribution:")
print(label_counts.to_string())
print()

positive_rate = df['is_declining'].mean()
print(f"Positive class (declining):  {df['is_declining'].sum():,} pages ({positive_rate:.1%})")
print(f"Negative class (not declining): {(df['is_declining']==0).sum():,} pages ({1-positive_rate:.1%})")
print()
print("Note: class imbalance is expected — most pages are not declining at any moment.")
print("This means accuracy is a poor metric; Precision@K and average precision are better.")

=== Target / proxy label: is_declining ===

trend_direction distribution:
trend_direction
down      16262
stable     5962
up         4388
new        2236
flat       1152

Positive class (declining):  16,262 pages (54.2%)
Negative class (not declining): 13,738 pages (45.8%)

Note: class imbalance is expected — most pages are not declining at any moment.
This means accuracy is a poor metric; Precision@K and average precision are better.


## 3. Success metric

**Primary metric: Precision@50**

Precision@K asks: of the top K pages the model flags as highest priority, how many are genuinely positive cases? This matches how the output is actually used — a reviewer works through the ranked list top-to-bottom, and the team can act on roughly 50 pages per week.

- Precision@50 = (true positives in the top 50) / 50
- Baseline rule score: **0.240** — about 12 of the top 50 are real positives
- Random forest (starter): **0.740** — about 37 of the top 50 are real positives

**Secondary metrics:**
- **ROC AUC** — overall ability to separate positives from negatives across all thresholds
- **Average precision** — area under the precision-recall curve; useful when positives are rare
- **Top-20 manual review** — reading the actual top 20 pages and checking whether the reason codes make sense to a human

**Why not accuracy?** The label is imbalanced — if only 20% of pages are declining, a model that always predicts 'not declining' gets 80% accuracy and is useless. Precision@K forces the model to be right about the pages it's most confident about.

**Why not recall?** Maximising recall would flag almost everything. The constraint is reviewer capacity — a list of 5,000 pages is no better than no list at all.

**The honest bar:** Beat the baseline's Precision@50 of 0.240 on a held-out client set. The starter random forest hits 0.740 on the starter slice; the goal for the full capstone is to earn a similar result on the warehouse data with proper time-aware or client-holdout validation.

In [4]:
import pandas as pd
import numpy as np

# Reproduce the verified starter model results for reference
results = {
    'Method': ['Baseline rules', 'Logistic regression', 'Decision tree', 'Random forest'],
    'ROC AUC': [0.627, 0.700, 0.742, 0.750],
    'Avg Precision': [0.468, 0.522, 0.575, 0.618],
    'Precision@50': [0.240, 0.400, 0.540, 0.740]
}

results_df = pd.DataFrame(results)
print("=== Verified starter model results (from outputs/model_results.json) ===")
print(results_df.to_string(index=False))
print()

baseline_p50 = 0.240
rf_p50 = 0.740
print(f"Precision@50 lift from baseline to random forest: +{rf_p50 - baseline_p50:.3f}")
print(f"In reviewer terms: {int(baseline_p50*50)} correct recommendations → {int(rf_p50*50)} correct, per 50 pages reviewed")
print()
print("Target for capstone: match or exceed Precision@50 = 0.740")
print("Validation: client-holdout (whole clients kept out of training)")

=== Verified starter model results (from outputs/model_results.json) ===
             Method  ROC AUC  Avg Precision  Precision@50
     Baseline rules    0.627          0.468          0.24
Logistic regression    0.700          0.522          0.40
      Decision tree    0.742          0.575          0.54
      Random forest    0.750          0.618          0.74

Precision@50 lift from baseline to random forest: +0.500
In reviewer terms: 12 correct recommendations → 37 correct, per 50 pages reviewed

Target for capstone: match or exceed Precision@50 = 0.740
Validation: client-holdout (whole clients kept out of training)


## 4. The unit of analysis, as a real dataframe

**One row = one content page, scored at a point in time using signals from the prior 90 days.**

The grain is the content item. Every row represents a single page belonging to a single client. The features describe what happened to that page over the past 90 days (impressions, clicks, sessions, position, engagement). The label says whether that page is currently on a downward trend.

Key columns in the unit of analysis:
- `content_id` — anonymised page identifier (join key, not a feature)
- `client_id` — anonymised client identifier (used for grouping and holdout, not a feature)
- `impressions_90d`, `clicks_90d`, `sessions_90d` — volume signals
- `avg_position`, `ctr` — search performance signals
- `content_age_days`, `word_count` — content metadata signals
- `trend_direction` → `is_declining` — the proxy label

In [5]:
import pandas as pd
import numpy as np

df = pd.read_csv('../../data/raw/content_refresh_anonymized.csv')
df['is_declining'] = (df['trend_direction'] == 'down').astype(int)

# Eligibility filter: enough signal to be worth scoring
eligible = df[(df['impressions_90d'] > 0) & (df['content_age_days'] >= 90)].copy()
eligible = eligible.drop_duplicates(subset='content_id')

# Select key columns to show the unit of analysis clearly
display_cols = [
    'content_id', 'client_id',
    'impressions_90d', 'clicks_90d', 'sessions_90d',
    'avg_position', 'ctr',
    'content_age_days', 'word_count',
    'trend_direction', 'is_declining'
]

# Show columns that actually exist
available_cols = [c for c in display_cols if c in eligible.columns]
unit_df = eligible[available_cols].head(10).reset_index(drop=True)

print("=== Unit of analysis: one row = one content page ===")
print(f"Eligible pages (after filter): {len(eligible):,}")
print()
print(unit_df.to_string())
print()
print("Each row is a page. 'is_declining' is the target column (1 = declining, 0 = not).")
print("Features are everything to the left of trend_direction.")

=== Unit of analysis: one row = one content page ===
Eligible pages (after filter): 30,000

             content_id          client_id  impressions_90d  clicks_90d  sessions_90d  avg_position   ctr  content_age_days  word_count trend_direction  is_declining
0  content_304f48230142  client_f369cb89fc             3803          29            17          10.6  0.76               187      3221.0            down             1
1  content_a1fb4e703a9e  client_4e07408562            15320           7             9          20.3  0.05               445      2481.0            down             1
2  content_9aa793d4d895  client_7f2253d7e2            12581          11            11          36.5  0.09               141      3515.0            down             1
3  content_331d6c4de07b  client_19581e27de            11751          58            78           6.2  0.49               463         NaN          stable             0
4  content_d99b7a2d90ca  client_3fdba35f04            19140          24       

In [6]:
import pandas as pd
import numpy as np

df = pd.read_csv('../../data/raw/content_refresh_anonymized.csv')
df['is_declining'] = (df['trend_direction'] == 'down').astype(int)
eligible = df[(df['impressions_90d'] > 0) & (df['content_age_days'] >= 90)].copy()

# Sketch what the target column looks like across the feature space
print("=== Target column sketch: is_declining by impression tier ===")
if 'impression_tier' in eligible.columns:
    sketch = eligible.groupby('impression_tier')['is_declining'].agg(['mean','sum','count'])
    sketch.columns = ['decline_rate', 'declining_pages', 'total_pages']
    print(sketch.round(3).to_string())
else:
    # Build a simple tier manually
    eligible['imp_tier'] = pd.cut(
        eligible['impressions_90d'],
        bins=[0, 100, 500, 2000, float('inf')],
        labels=['low (0-100)', 'medium (100-500)', 'high (500-2000)', 'very high (2000+)']
    )
    sketch = eligible.groupby('imp_tier', observed=True)['is_declining'].agg(['mean','sum','count'])
    sketch.columns = ['decline_rate', 'declining_pages', 'total_pages']
    print(sketch.round(3).to_string())

print()
print("Decline rate varies by impression volume — this signal variation is what the model learns from.")

=== Target column sketch: is_declining by impression tier ===
                 decline_rate  declining_pages  total_pages
impression_tier                                            
excellent               0.462              498         1078
good                    0.586             4223         7205
low                     0.454             5106        11248
moderate                0.615             6435        10469

Decline rate varies by impression volume — this signal variation is what the model learns from.


## 5. Why ML beats a fixed rule here

**The fixed rule problem:** The baseline score is a weighted sum of four components with hand-tuned weights (0.40 visibility + 0.30 freshness risk + 0.25 position opportunity + 0.05 depth gap). These weights were chosen by a human based on intuition about what matters. They treat every page the same way, regardless of what combination of signals that page actually shows.

**Where the rule breaks down:**
- A page could be old AND have high impressions AND have a declining trend AND have low CTR — four signals pointing the same direction. The rule scores each independently and adds them up. A model can learn that *this specific combination* is a much stronger signal than any single factor alone.
- A page could be fresh but declining fast. The freshness rule would underweight it; a model would see the declining trend as the dominant signal.
- A page could be stale but stable. The freshness rule would over-flag it; a model could learn that stale-but-stable pages rarely need urgent review.

**What the data proves:** The starter results show a Precision@50 jump from 0.240 (baseline) to 0.740 (random forest) — on the same data, same validation, same task. That 3× lift means the model is finding real signal that the fixed weights miss.

**The core claim:** ML is justified here not because the problem is complex, but because the *interaction between signals* is more predictive than any single signal alone, and that interaction is learnable from data. The fixed rule cannot capture it; a model can.

**Where human judgment still wins:** Choosing which pages to actually act on, deciding what 'refresh' means for a specific piece of content, and catching edge cases (seasonal content, recently-published pages, pages being restructured). The model ranks; the human decides.

In [7]:
import pandas as pd
import numpy as np

df = pd.read_csv('../../data/raw/content_refresh_anonymized.csv')
df['is_declining'] = (df['trend_direction'] == 'down').astype(int)
eligible = df[(df['impressions_90d'] > 0) & (df['content_age_days'] >= 90)].copy()

print("=== Why ML beats a fixed rule: signal interaction example ===")
print()

# Show that decline rate varies by combinations of signals, not just one
if 'avg_position' in eligible.columns and 'ctr' in eligible.columns:
    eligible['position_good'] = eligible['avg_position'].between(1, 10)
    eligible['ctr_low'] = eligible['ctr'] < eligible['ctr'].median()

    combo = eligible.groupby(['position_good', 'ctr_low'], observed=True)['is_declining'].agg(['mean', 'count'])
    combo.columns = ['decline_rate', 'page_count']
    combo.index = [
        f"position_good={a}, ctr_low={b}" for a, b in combo.index
    ]
    print("Decline rate by signal combination (position quality x CTR):")
    print(combo.round(3).to_string())
    print()
    print("The decline rate changes based on the COMBINATION of signals.")
    print("A fixed rule with independent weights can't capture this interaction.")
    print("A model learns which combinations are most predictive.")
else:
    # Fallback: show variance in decline rate across age tiers
    eligible['age_tier'] = pd.cut(eligible['content_age_days'],
                                   bins=[0, 180, 365, 730, float('inf')],
                                   labels=['<6mo', '6-12mo', '1-2yr', '2yr+'])
    tier_stats = eligible.groupby('age_tier', observed=True)['is_declining'].agg(['mean','count'])
    tier_stats.columns = ['decline_rate', 'pages']
    print("Decline rate by content age tier:")
    print(tier_stats.round(3).to_string())
    print()
    print("Decline rate varies across tiers — but age alone doesn't explain everything.")
    print("A model combines age with impressions, CTR, position etc. to find the real pattern.")

print()
print("Starter Precision@50 summary:")
print("  Baseline (fixed weights): 0.240  → 12/50 correct")
print("  Random forest (learned):  0.740  → 37/50 correct")
print("  Conclusion: the interaction between signals is learnable and valuable.")

=== Why ML beats a fixed rule: signal interaction example ===

Decline rate by signal combination (position quality x CTR):
                                    decline_rate  page_count
position_good=False, ctr_low=False         0.569        7356
position_good=False, ctr_low=True          0.492        9753
position_good=True, ctr_low=False          0.559        7834
position_good=True, ctr_low=True           0.574        5057

The decline rate changes based on the COMBINATION of signals.
A fixed rule with independent weights can't capture this interaction.
A model learns which combinations are most predictive.

Starter Precision@50 summary:
  Baseline (fixed weights): 0.240  → 12/50 correct
  Random forest (learned):  0.740  → 37/50 correct
  Conclusion: the interaction between signals is learnable and valuable.


## Self-check

Before you submit, confirm each line honestly:

- [x] Named the ML task type: supervised binary classification with ranking output
- [x] Named the target/proxy: `is_declining = (trend_direction == 'down')` with a note on the stronger capstone label
- [x] Named the success metric: Precision@50, with ROC AUC and average precision as secondary
- [x] Showed the unit of analysis as a real dataframe: one row = one content page
- [x] Explained why ML beats a fixed rule: signal interactions, backed by Precision@50 numbers
- [x] Tied the output to a real content action: ranked review queue, reviewer works top-to-bottom
- [x] No client names, URLs, or private queries anywhere
- [x] Claims use careful words: observed, directional, decision-support
- [x] Notebook runs top to bottom with no errors (Runtime → Run all)
- [x] Committed to repo under `work/notebooks/w02_ml_task_framing.ipynb`